# Financial News Sentiment Prediction using Deep Learning & BERT

This notebook follows the supplied project brief and uses the provided train/validation CSV files.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = Path(".")
TRAIN_PATH = BASE.parent / "sent_train(1).csv"
VALID_PATH = BASE.parent / "sent_valid(1).csv"

LABEL_MAP = {0: "Bearish", 1: "Bullish", 2: "Neutral"}


## 2. Load and validate data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
display(train_df.head())


In [ ]:
assert list(train_df.columns) == ["text", "label"]
assert list(valid_df.columns) == ["text", "label"]
assert train_df.isna().sum().sum() == 0
assert valid_df.isna().sum().sum() == 0
assert set(train_df["label"].unique()) <= {0, 1, 2}
assert set(valid_df["label"].unique()) <= {0, 1, 2}
assert len(set(train_df["text"]) & set(valid_df["text"])) == 0
print("Basic data checks passed.")


## 3. Exploratory data analysis

In [ ]:
def class_table(df):
    out = df["label"].value_counts().sort_index().rename("count").to_frame()
    out["sentiment"] = [LABEL_MAP[i] for i in out.index]
    out["percent"] = (100 * out["count"] / len(df)).round(2)
    return out

display(class_table(train_df))
display(class_table(valid_df))


In [ ]:
train_df["word_count"] = train_df["text"].astype(str).str.split().str.len()
valid_df["word_count"] = valid_df["text"].astype(str).str.split().str.len()

print(train_df["word_count"].describe())
print(valid_df["word_count"].describe())


## 4. Baseline preprocessing
Light cleaning removes URLs, normalizes mentions and whitespace, while preserving tickers and finance terms.

In [ ]:
from src.data import clean_text, build_vocab, SentimentDataset

train_df["clean_text"] = train_df["text"].map(clean_text)
valid_df["clean_text"] = valid_df["text"].map(clean_text)

vocab = build_vocab(train_df["clean_text"], max_size=30000)
print("Vocabulary size:", len(vocab))


## 5. Train Simple RNN, LSTM and GRU

The project implementation will select the best baseline using validation macro F1.

In [ ]:
# Training cell intentionally separated so it can be run when PyTorch is available.
import torch
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight

from src.models import RNNClassifier
from src.training import train_model, evaluate

MAX_LEN = 80
BATCH_SIZE = 64
train_ds = SentimentDataset(train_df, vocab, MAX_LEN)
valid_ds = SentimentDataset(valid_df, vocab, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classes = np.array([0, 1, 2])
weights = compute_class_weight("balanced", classes=classes, y=train_df["label"])
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))

baseline_results = {}
for kind in ["rnn", "lstm", "gru"]:
    model = RNNClassifier(len(vocab), 128, 128, 3, rnn_type=kind).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model, history = train_model(model, train_loader, valid_loader, optimizer, criterion, device)
    metrics = evaluate(model, valid_loader, criterion, device)
    baseline_results[kind] = metrics
    print(kind.upper(), "accuracy=", round(metrics["accuracy"], 4), "macro_f1=", round(metrics["macro_f1"], 4))


## 6. BERT / FinBERT fine-tuning

Use the same official training and validation split. Transformer tokenization should remain separate from the baseline vocabulary.

In [ ]:
# Run when a pretrained checkpoint is available.
# Suggested finance-specific starting point: a FinBERT-compatible 3-class sequence classifier.
#
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, ...
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
#
# Fine-tune only on train_df and evaluate only on valid_df.


## 7. Model comparison and error analysis

Compare accuracy, macro F1, per-class precision/recall and confusion matrices. Inspect misclassified Bearish/Bullish examples rather than reporting only aggregate metrics.

## 8. Inference and Streamlit

Export the best model, vocabulary/tokenizer and label mapping for the dashboard.